In [1]:
%pip install --upgrade --user --quiet google-cloud-aiplatform googlemaps

In [2]:
import IPython
import sys
import pprint

app = IPython.Application.instance()
app.kernel.do_shutdown(True)


if "google.colab" in sys.modules:
    from google.colab import auth



PROJECT_ID = ""  # @param hide {type:"string"}
LOCATION = ""  # @param {type:"string"}

import vertexai

vertexai.init(project=PROJECT_ID, location=LOCATION)

In [1]:
import json

from vertexai import generative_models
from vertexai.generative_models import GenerationConfig, GenerativeModel, Part, Image

response_schema = {
    'type': 'object',
    'properties': {
        'streetview': {
            'type': 'object',
            'properties': {
                'percentage_of_house_visible': {
                    'type': 'integer'
                }
            }
        },
        'architecture': {
            'type': 'object',
            'properties': {
                'style': {
                  'type': 'string'
                },
                'features': {
                    'type': 'array',
                    'items': {
                        'type': 'string'
                    }
                }
            }
        },
        'siding': {
            'type': 'object',
            'properties': {
                'color': { 'type': 'string' },
                'material': {
                    'type': 'string',
                    'enum': [
                        'brick',
                        'wood plank',
                        'wood shingle',
                        'vinyl',
                        'aluminum',
                        'stucco',
                        'stone',
                        'concrete',
                        'composite',
                        'unknown'
                    ]
                },
                'condition': {
                    'type': 'string',
                    'enum': [
                        'poor',
                        'fair',
                        'good'
                    ]
                }
            }
        },
        'roof': {
            'type': 'object',
            'properties': {
                'color': {
                    'type': 'string',
                    'enum': [
                        'red',
                        'gray',
                        'black',
                        'tan',
                        'green',
                        'other'
                    ]
                },
                'material': {
                    'type': 'string',
                    'enum': [
                        'asphalt',
                        'metal',
                        'slate',
                        'wood'
                    ]
                },
                'shape': {
                    'type': 'string',
                    'enum': [
                        'gable',
                        'hip',
                        'gambrel',
                        'flat'
                    ]
                },
                'dormers': {
                    'type': 'integer'
                },
                'chimneys': {
                    'type': 'integer'
                },
                'gutters': {
                    'type': 'string'
                },
                'penetrations': {
                    'type': 'integer'
                },
                'condition': {
                    'type': 'string'
                }
            }
        },
        'building_features': {
            'type': 'array',
            'items': {
                'type': 'object',
                'properties': {
                    'feature': {
                        'type': 'string',
                        'enum': [
                            'garage',
                            'carport',
                            'shed',
                            'gazebo',
                            'porch',
                            'deck'
                        ]
                    },
                    'description': {
                        'type': 'string'
                    }
                }
            }
        },
        'landscape_features': {
            'type': 'array',
            'items': {
                'type': 'object',
                'properties': {
                    'feature': {
                        'type': 'string',
                        'enum': [
                            'driveway',
                            'garden',
                            'sidewalk',
                            'patio',
                            'ditch',
                            'swale',
                            'mailbox'
                        ]
                    },
                    'description': { 'type': 'string' }
                }
            }
        },
        'trees': {
            'type': 'array',
            'items': {
                'type': 'object',
                'properties': {
                    'estimated_species': {
                        'type': 'string'
                    },
                    'size': {
                        'type': 'string'
                    },
                    'condition': {
                        'type': 'string'
                    },
                    'location': {
                        'type': 'string'
                    }
                }
            }
        },
        'hazards': {
            'type': 'array',
            'items': {
                'type': 'string',
                'enum': [
                    'in ground pool',
                    'above ground pool',
                    'tree hanging over roof',
                    'near highway',
                    'body of water'
                ]
            }
        },
        'road_condition': {
            'type': 'string',
            'enum': [
                'poor',
                'fair',
                'good',
                'freshly paved'
            ]
        },
        'flooding': {
            'type': 'object',
            'properties': {
                'first_floor_distance_from_ground_feet': {
                    'type': 'integer'
                },
                'grading_from_street': {
                    'type': 'string'
                },
                'foundation_material': {
                    'type': 'string'
                }
            }
        },
        'real_estate_marketing_description': {
            'type': 'string'
        }
    },
    'required': [
        'streetview',
        'architecture',
        'siding',
        'roof',
        'hazards',
        'trees',
        'road_condition',
        'real_estate_marketing_description',
        'landscape_features',
        'building_features',
        'flooding'
    ]
}

generation_config = GenerationConfig(
    temperature=1,
    top_p=.1,
    top_k=32,
    candidate_count=1,
    max_output_tokens=8192,
    response_mime_type="application/json",
    response_schema=response_schema
)
model = GenerativeModel(
    model_name="gemini-1.5-pro",
    generation_config=generation_config,
    system_instruction=[
        "You analyze images of a residential property to identify details that would be useful to a potential buyer, owner, or insurance company.",
        "You will see two aerial images, and three streetview images of the subject property, at different zoom levels."

    ]
)

In [4]:
import requests
import googlemaps
from datetime import datetime
import pprint
from google.colab.patches import cv2_imshow
import cv2 as cv
from datetime import datetime

MAPS_API_KEY = '' # @param {type:"string"}

ADDRESS = "" # @param {type:"string"}

gmaps = googlemaps.Client(key=MAPS_API_KEY)

# Geocoding an address
geocode_result = gmaps.geocode(ADDRESS)[0]
#pprint.pprint(geocode_result)
lat = geocode_result['geometry']['location']['lat']
lng = geocode_result['geometry']['location']['lng']

aerial_iter1 = gmaps.static_map(
    size=(512, 512),
    zoom=19,
    center=(lat, lng),
    maptype="hybrid",
    format="png32",
    scale=1)
with open('aerial1.png', 'wb') as f:
  for chunk in aerial_iter1:
    f.write(chunk)

aerial_iter2 = gmaps.static_map(
    size=(512, 512),
    zoom=21,
    center=(lat, lng),
    maptype="hybrid",
    format="png32",
    scale=1)
with open('aerial2.png', 'wb') as f:
  for chunk in aerial_iter2:
    f.write(chunk)

sv1_url = 'https://maps.googleapis.com/maps/api/streetview?size=640x640&scale=2&location={}&fov=20&pitch=5&key={}'.format(ADDRESS, MAPS_API_KEY)
sv1_iter = requests.get(sv1_url).iter_content()
with open('streetview1.png', 'wb') as f:
  for chunk in sv1_iter:
    f.write(chunk)

sv2_url = 'https://maps.googleapis.com/maps/api/streetview?size=640x640&location={}&fov=80&pitch=-5&key={}'.format(ADDRESS, MAPS_API_KEY)
sv2_iter = requests.get(sv2_url).iter_content()
with open('streetview2.png', 'wb') as f:
  for chunk in sv2_iter:
    f.write(chunk)

sv3_url = 'https://maps.googleapis.com/maps/api/streetview?size=640x640&location={}&fov=120&pitch=5&key={}'.format(ADDRESS, MAPS_API_KEY)
sv3_iter = requests.get(sv3_url).iter_content()
with open('streetview3.png', 'wb') as f:
  for chunk in sv3_iter:
    f.write(chunk)

#cv2_imshow(cv.imread('aerial1.png'))
#cv2_imshow(cv.imread('aerial2.png'))
#cv2_imshow(cv.imread('streetview1.png'))
#cv2_imshow(cv.imread('streetview2.png'))
#cv2_imshow(cv.imread('streetview3.png'))

sv1 = Part.from_image(Image.load_from_file('streetview1.png'))
sv2 = Part.from_image(Image.load_from_file('streetview2.png'))
sv3 = Part.from_image(Image.load_from_file('streetview3.png'))
aerial1 = Part.from_image(Image.load_from_file('aerial1.png'))
aerial2 = Part.from_image(Image.load_from_file('aerial2.png'))

prompt_text = """
  The address of the subject property is {}. Analyze the subject property and populate the response_schema fields.

""".format(address)

prompt = [prompt_text, aerial1, aerial2, sv1, sv2, sv3]

import pprint

response = model.generate_content(prompt)

json_response = json.loads(response.text)
print(pprint.pprint(json_response))


{'architecture': {'features': ['covered porch', 'front facing dormer'],
                  'style': 'colonial'},
 'building_features': [{'description': 'covered front porch',
                        'feature': 'porch'},
                       {'description': 'detached garage, not visible in street '
                                       'view',
                        'feature': 'garage'}],
 'flooding': {'first_floor_distance_from_ground_feet': 3,
              'foundation_material': 'brick foundation visible',
              'grading_from_street': 'positive'},
 'hazards': ['tree hanging over roof'],
 'landscape_features': [{'description': 'concrete driveway',
                         'feature': 'driveway'},
                        {'description': 'concrete sidewalk',
                         'feature': 'sidewalk'},
                        {'description': 'landscaped garden beds in front of '
                                        'house',
                         'feature': 'garden'}]